# BirdCLEF+ 2026: experiment history

Every leaderboard submission and every SED training run, read back from MLflow
(experiments `birdclef-lb` and `birdclef-sed-train`). `track.py` writes them; this
notebook only reads.

Before running:

```
zsh MoleculeID-from-Mass-Spectra/mlflow_server.sh &                      # once per boot
MLFLOW_TRACKING_URI=http://127.0.0.1:5050 python3 BirdCLEF/track.py      # picks up new scores
```

UI: http://127.0.0.1:5050

In [ ]:
import os

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5050")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from matplotlib.lines import Line2D
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Categorical slots in fixed order (validated: adjacent CVD ΔE ≥ 9.1); text stays in ink.
FAMILY = {"birdnet": "#2a78d6", "perch": "#eb6834", "perch+sed": "#1baf7a", "sed": "#eda100"}
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlelocation": "left", "axes.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10, "lines.linewidth": 2,
    "legend.frameon": False,
})

## Leaderboard submissions

In [ ]:
lb = mlflow.search_runs(experiment_names=["birdclef-lb"], order_by=["attributes.start_time ASC"])
lb = lb.rename(columns={
    "tags.mlflow.runName": "submission", "tags.family": "family", "tags.status": "lb_status",
    "tags.submission_id": "submission_id", "metrics.lb_public": "public",
    "metrics.lb_private": "private", "metrics.local_cv": "local_cv", "params.w_sed": "w_sed",
    "params.sed_train": "sed_train", "params.kernel": "kernel", "params.kernel_version": "kernel_version",
})
lb["date"] = lb["start_time"].dt.strftime("%Y-%m-%d")
lb["w_sed"] = pd.to_numeric(lb["w_sed"])
PERCH = lb.loc[lb["family"] == "perch", "private"].iloc[0]

(lb[["date", "submission", "family", "lb_status", "public", "private", "local_cv", "submission_id"]]
 .style.format({"public": "{:.5f}", "private": "{:.5f}", "local_cv": "{:.4f}"}, na_rep="–")
 .hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(lb) + 1.2))
ys = list(range(len(lb)))[::-1]                      # oldest at the top
for y, (_, r) in zip(ys, lb.iterrows()):
    c = FAMILY[r.family]
    ax.plot(r.public, y, "o", ms=8, mfc=SURFACE, mec=c, mew=1.8)
    ax.plot(r.private, y, "o", ms=9.5, color=c, mec=SURFACE, mew=1.2)
ax.set_yticks(ys, [s + ("  (pending)" if st == "PENDING" else "") for s, st in zip(lb.submission, lb.lb_status)])
ax.grid(axis="y", visible=False)
ax.axvline(PERCH, color=INK2, lw=1, ls="--")
ax.annotate(f"Perch kernel, best private {PERCH:.5f}", (PERCH, ys[0] + 0.6), xytext=(-4, 0),
            textcoords="offset points", ha="right", va="center", fontsize=9, color=INK2, annotation_clip=False)
handles = [Line2D([], [], ls="", marker="o", ms=8, color=c, label=f) for f, c in FAMILY.items()]
handles += [Line2D([], [], ls="", marker="o", ms=8, color=INK2, label="private"),
            Line2D([], [], ls="", marker="o", ms=8, mfc=SURFACE, mec=INK2, mew=1.8, label="public")]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
ax.set_xlabel("leaderboard macro ROC-AUC")
ax.set_title("Every submission, oldest first")
plt.show()

## SED training runs

Fold-0 validation AUC per epoch. It is inflated and was useless as a guide: every
teacher had seen fold 0's labels (RESULTS.md section 16).

In [ ]:
train = mlflow.search_runs(experiment_names=["birdclef-sed-train"]).set_index("tags.mlflow.runName")
train = train.loc[[k for k in ("sed-A", "sed-B", "sed-Bfull") if k in train.index]]
curves = {k: pd.Series({m.step: m.value for m in client.get_metric_history(rid, "val_auc")}).sort_index()
          for k, rid in train["run_id"].items()}
LABEL = {"sed-A": "pilot A: 12k windows, public teacher", "sed-B": "pilot B: 12k windows, Perch teacher",
         "sed-Bfull": "full B: 127k windows, Perch teacher"}

train[["params.n_pseudo", "params.n_train", "metrics.val_auc_final", "metrics.val_auc_max",
       "metrics.gpu_min", "metrics.decode_min", "params.teacher"]].rename_axis("run")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ends = sorted(((s.iloc[-1], k) for k, s in curves.items()), reverse=True)
label_y, last = {}, 2.0
for v, k in ends:                                    # push end labels down until they clear each other
    last = min(v, last - 0.025)
    label_y[k] = last
for (k, s), c in zip(curves.items(), SERIES):
    ax.plot(s.index, s.values, color=c, label=LABEL[k])
    ax.annotate(f"{k} {s.iloc[-1]:.4f}", (s.index[-1], s.iloc[-1]), xytext=(s.index[-1] + 0.6, label_y[k]),
                textcoords="data", va="center", fontsize=9, color=INK, annotation_clip=False)
ax.set_xlim(right=max(s.index[-1] for s in curves.values()) + 4.5)
ax.set_xticks(range(0, 20, 2))
ax.set_ylim(0.5, 1.02)
ax.set_xlabel("epoch")
ax.set_ylabel("fold-0 val ROC-AUC")
ax.legend(loc="lower right", fontsize=9)
ax.set_title("Fold-0 validation AUC per epoch (inflated: every teacher saw these labels)")
plt.show()

## Does data scale make the SED pay?

Pilot B and full B differ only in how many unlabelled windows carry Perch-teacher
targets: 12,000 against all 127,104.

In [ ]:
def private(train_key, family, w=None):
    m = (lb["sed_train"] == train_key) & (lb["family"] == family)
    if w is not None:
        m &= lb["w_sed"] == w
    return lb.loc[m, "private"].iloc[0]

scale = pd.DataFrame({
    "SED alone": [private("sed-B", "sed"), private("sed-Bfull", "sed")],
    "Perch + SED, w=0.5": [private("sed-B", "perch+sed", 0.5), private("sed-Bfull", "perch+sed", 0.5)],
}, index=["pilot B\n12,000 windows", "full B\n127,104 windows"])
print((scale.iloc[1] - scale.iloc[0]).map("{:+.4f}".format).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4.2))
x = [0, 1]
for (name, col), c in zip(scale.items(), SERIES):
    ax.plot(x, col.values, "-o", color=c, ms=8, mec=SURFACE, mew=1.5, label=name)
    for xi, v in zip(x, col.values):
        ax.annotate(f"{v:.4f}", (xi, v), xytext=(-10 if xi == 0 else 10, 0), textcoords="offset points",
                    ha="right" if xi == 0 else "left", va="center", fontsize=9, color=INK)
ax.axhline(PERCH, color=INK2, lw=1, ls="--")
ax.annotate(f"Perch alone {PERCH:.4f}", (0.5, PERCH), xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=9, color=INK2)
ax.set_xticks(x, scale.index)
ax.set_xlim(-0.45, 1.45)
ax.grid(axis="x", visible=False)
ax.set_ylabel("private leaderboard")
ax.legend(loc="lower right", fontsize=9)
ax.set_title("10x the windows: SED alone +0.031, blend still below Perch")
plt.show()

## Blend weight, full B

`w_sed = 0` is the Perch kernel itself (the blend kernel's Perch half is that kernel
unchanged); `w_sed = 1` is the SED alone. A submission still scoring is left out.

In [ ]:
full = lb[lb["sed_train"] == "sed-Bfull"].copy()
full.loc[full["family"] == "sed", "w_sed"] = 1.0
curve = pd.concat([pd.DataFrame({"w_sed": [0.0], "private": [PERCH], "public": [lb.loc[lb.family == "perch", "public"].iloc[0]]}),
                   full[["w_sed", "private", "public"]]]).dropna().sort_values("w_sed")
display(curve.set_index("w_sed"))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(curve.w_sed, curve.private, "-o", color=SERIES[0], ms=8, mec=SURFACE, mew=1.5)
best = curve.loc[curve.private.idxmax()]
ax.annotate(f"best {best.private:.5f} at w={best.w_sed:g}", (best.w_sed, best.private), xytext=(10, -4),
            textcoords="offset points", va="top", fontsize=9, color=INK)
ax.set_xticks(curve.w_sed)
ax.set_xlabel("w_sed (0 = Perch only, 1 = SED only)")
ax.set_ylabel("private leaderboard")
ax.set_title("Private score against the SED's blend weight")
plt.show()